## Model Refinement for Survival Analysis

### Objective
This notebook refines the baseline Cox Proportional Hazards model
to improve robustness, generalization, and interpretability.

Main goals:
- evaluate train / validation performance
- compare baseline and penalized Cox models
- quantify overfitting risk
- identify a more stable modeling strategy

---

## 목적
이 노트북의 목적은 baseline Cox Proportional Hazards 모델을 개선하여
안정성, 일반화 성능, 해석 가능성을 높이는 것이다.

핵심 목표:
- train / validation 성능 평가
- baseline Cox와 penalized Cox 비교
- 과적합 위험 정량화
- 더 안정적인 모델링 전략 도출

In [ ]:
# 1. Import
import pandas as pd
import numpy as np

from lifelines import CoxPHFitter
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [78]:
# 2. Reproducibility
np.random.seed(42)

In [ ]:
# 3. Load data
file_path = "../data/processed/metabric_clinical_featurized.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

Shape: (1353, 12)


,Age at Diagnosis,Lymph nodes examined positive,Tumor Size,Neoplasm Histologic Grade_2.0,Neoplasm Histologic Grade_3.0,Tumor Stage_2.0,Tumor Stage_3.0,Tumor Stage_4.0,ER Status_Positive,HER2 Status_Positive,time,event
0,75.6500,10.0000,22.0000,0,1,1,0,0,1,0,140.5000,0
1,43.1900,0.0000,10.0000,0,1,0,0,0,1,0,84.6333,0
2,48.8700,1.0000,15.0000,1,0,1,0,0,1,0,163.7000,1
3,47.6800,3.0000,25.0000,1,0,1,0,0,1,0,164.9333,0
4,76.9700,8.0000,40.0000,0,1,1,0,0,1,0,41.3667,1


## Data Splitting and Reproducibility

To ensure reproducibility, a fixed random seed is used.

The dataset is split into training and validation sets
with stratification on the event variable so that
the event proportion remains similar across both sets.

---

## 데이터 분할 및 재현성

재현성을 확보하기 위해 고정된 random seed를 사용한다.

또한 event 변수의 비율이 train / validation 양쪽에서 유사하게 유지되도록
stratified split을 적용한다.

In [ ]:
# 4. Train/Validation Split
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["event"]
)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)

print("\nTrain event distribution:")
print(train_df["event"].value_counts(normalize=True))

print("\nValidation event distribution:")
print(valid_df["event"].value_counts(normalize=True))

## Baseline Cox Proportional Hazards Model

We first fit the standard Cox PH model on the training set.

This serves as the baseline interpretable survival model.

---

## Baseline Cox Proportional Hazards 모델

먼저 training set에 기본 Cox PH 모델을 적합한다.

이 모델은 해석 가능한 baseline survival model 역할을 한다.

In [80]:
# 5. Fit Baseline Cox model
baseline_cph = CoxPHFitter()
baseline_cph.fit(train_df, duration_col="time", event_col="event")

baseline_valid_cindex = baseline_cph.score(
    valid_df,
    scoring_method="concordance_index"
)

print("Baseline train concordance:", round(baseline_cph.concordance_index_, 4))
print("Baseline validation concordance:", round(baseline_valid_cindex, 4))

Baseline train concordance: 0.7157
Baseline validation concordance: 0.629


In [81]:
# 6. Baseline Summary
baseline_summary = baseline_cph.summary.sort_values(by="p")
baseline_summary

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
Lymph nodes examined positive,0.0511,1.0524,0.0118,0.0280,0.0743,1.0283,1.0771,0.0000,4.3260,0.0000,16.0071
Tumor Stage_2.0,0.5127,1.6698,0.1366,0.2450,0.7804,1.2776,2.1824,0.0000,3.7538,0.0002,12.4870
Tumor Stage_3.0,0.7509,2.1189,0.2425,0.2756,1.2261,1.3174,3.4080,0.0000,3.0967,0.0020,8.9973
HER2 Status_Positive,0.4327,1.5414,0.1430,0.1525,0.7129,1.1647,2.0400,0.0000,3.0265,0.0025,8.6587
Neoplasm Histologic Grade_3.0,0.8279,2.2885,0.2826,0.2740,1.3817,1.3153,3.9819,0.0000,2.9297,0.0034,8.2033
Tumor Size,0.0083,1.0083,0.0031,0.0022,0.0144,1.0022,1.0145,0.0000,2.6514,0.0080,6.9628
ER Status_Positive,-0.3430,0.7096,0.1324,-0.6025,-0.0836,0.5475,0.9198,0.0000,-2.5919,0.0095,6.7111
Tumor Stage_4.0,1.2290,3.4179,0.4821,0.2842,2.1739,1.3287,8.7923,0.0000,2.5494,0.0108,6.5341
Neoplasm Histologic Grade_2.0,0.5260,1.6921,0.2838,-0.0302,1.0822,0.9702,2.9512,0.0000,1.8535,0.0638,3.9701


## Penalized Cox Model

Next, we fit a penalized Cox model using L2-style regularization.

This is useful when:
- some subgroup sizes are small
- mild multicollinearity may exist
- coefficient stability is important

---

## Penalized Cox 모델

다음으로 regularization이 적용된 penalized Cox 모델을 적합한다.

이 접근은 다음 상황에서 유용하다.
- 일부 subgroup의 표본 수가 작을 때
- 약한 다중공선성이 존재할 수 있을 때
- 계수 안정성이 중요할 때

In [82]:
# 7. Fit Penalized Cox Model
penalized_cph = CoxPHFitter(penalizer=0.1)
penalized_cph.fit(train_df, duration_col="time", event_col="event")

penalized_valid_cindex = penalized_cph.score(
    valid_df,
    scoring_method="concordance_index"
)

print("Penalized train concordance:", round(penalized_cph.concordance_index_, 4))
print("Penalized validation concordance:", round(penalized_valid_cindex, 4))

Penalized train concordance: 0.7183
Penalized validation concordance: 0.6384


In [83]:
# 8. Penalized Summary
penalized_summary = penalized_cph.summary.sort_values(by="p")
penalized_summary

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
Lymph nodes examined positive,0.0498,1.0510,0.0106,0.0290,0.0705,1.0294,1.0730,0.0000,4.6999,0.0000,18.5516
Tumor Size,0.0090,1.0091,0.0027,0.0037,0.0143,1.0037,1.0144,0.0000,3.3353,0.0009,10.1970
Tumor Stage_2.0,0.3295,1.3902,0.1042,0.1252,0.5337,1.1334,1.7053,0.0000,3.1615,0.0016,9.3155
HER2 Status_Positive,0.3913,1.4789,0.1286,0.1392,0.6433,1.1494,1.9028,0.0000,3.0424,0.0023,8.7351
Tumor Stage_3.0,0.5323,1.7029,0.1892,0.1615,0.9031,1.1753,2.4673,0.0000,2.8136,0.0049,7.6733
ER Status_Positive,-0.3072,0.7355,0.1135,-0.5296,-0.0848,0.5888,0.9187,0.0000,-2.7072,0.0068,7.2032
Neoplasm Histologic Grade_3.0,0.3256,1.3848,0.1283,0.0741,0.5770,1.0769,1.7807,0.0000,2.5377,0.0112,6.4857
Tumor Stage_4.0,1.0042,2.7296,0.4492,0.1237,1.8846,1.1317,6.5839,0.0000,2.2353,0.0254,5.2992
Age at Diagnosis,0.0041,1.0041,0.0039,-0.0036,0.0117,0.9964,1.0118,0.0000,1.0475,0.2949,1.7618


## Validation Performance

Validation concordance is used to assess how well each model generalizes
to unseen data.

A stronger model should not only fit the training data well,
but also maintain good ranking performance on the validation set.

---

## Validation 성능

Validation concordance는 모델이 보지 않은 데이터에 대해 얼마나 잘 일반화되는지 평가하기 위한 지표이다.

좋은 모델은 training set에만 잘 맞는 것이 아니라,
validation set에서도 안정적인 ranking 성능을 보여야 한다.

In [84]:
# 9. Model Performance Comparison
model_performance = pd.DataFrame({
    "model": ["baseline_cox", "penalized_cox"],
    "train_cindex": [
        baseline_cph.concordance_index_,
        penalized_cph.concordance_index_
    ],
    "valid_cindex": [
        baseline_valid_cindex,
        penalized_valid_cindex
    ]
})

model_performance["overfitting_gap"] = (
    model_performance["train_cindex"] - model_performance["valid_cindex"]
)

model_performance = model_performance.sort_values(
    by="valid_cindex",
    ascending=False
).reset_index(drop=True)

model_performance

,model,train_cindex,valid_cindex,overfitting_gap
0,penalized_cox,0.7183,0.6384,0.0799
1,baseline_cox,0.7157,0.6290,0.0867


## Model Performance Comparison

We compare the models using:
- training concordance
- validation concordance
- overfitting gap

Interpretation:
- Higher validation concordance is preferred.
- A smaller overfitting gap indicates better stability and generalization.

---

## 모델 성능 비교 기준

다음 기준으로 모델을 비교한다.
- training concordance
- validation concordance
- overfitting gap

해석 기준:
- validation concordance가 높을수록 좋다.
- overfitting gap이 작을수록 더 안정적이고 일반화 성능이 좋다고 볼 수 있다.

In [85]:
# 10. Best Model Selection
best_model = model_performance.loc[
    model_performance["valid_cindex"].idxmax()
]

print("Best model based on validation concordance:", best_model["model"])
print("Best validation concordance index:", round(best_model["valid_cindex"], 4))
print("Overfitting gap:", round(best_model["overfitting_gap"], 4))

Best model based on validation concordance: penalized_cox
Best validation concordance index: 0.6384
Overfitting gap: 0.0799


## Best Model Selection

The final model is selected primarily based on validation concordance.

The overfitting gap is also reviewed to determine whether
the model is stable enough for interpretation and reporting.

---

## 최종 모델 선택

최종 모델은 우선적으로 validation concordance를 기준으로 선택한다.

추가적으로 overfitting gap을 함께 검토하여
해석 및 보고에 적합할 정도로 안정적인 모델인지 판단한다.

In [86]:
# 11. Coefficient Stability Comparison
comparison_df = pd.DataFrame({
    "feature": baseline_cph.summary.index,
    "baseline_coef": baseline_cph.summary["coef"].values,
    "baseline_hr": baseline_cph.summary["exp(coef)"].values,
    "penalized_coef": penalized_cph.summary["coef"].reindex(baseline_cph.summary.index).values,
    "penalized_hr": penalized_cph.summary["exp(coef)"].reindex(baseline_cph.summary.index).values,
})

comparison_df

,feature,baseline_coef,baseline_hr,penalized_coef,penalized_hr
0,Age at Diagnosis,0.0059,1.0059,0.0041,1.0041
1,Lymph nodes examined positive,0.0511,1.0524,0.0498,1.0510
2,Tumor Size,0.0083,1.0083,0.0090,1.0091
3,Neoplasm Histologic Grade_2.0,0.5260,1.6921,0.0257,1.0260
4,Neoplasm Histologic Grade_3.0,0.8279,2.2885,0.3256,1.3848
5,Tumor Stage_2.0,0.5127,1.6698,0.3295,1.3902
6,Tumor Stage_3.0,0.7509,2.1189,0.5323,1.7029
7,Tumor Stage_4.0,1.2290,3.4179,1.0042,2.7296
8,ER Status_Positive,-0.3430,0.7096,-0.3072,0.7355
9,HER2 Status_Positive,0.4327,1.5414,0.3913,1.4789


In [87]:
# 12. Focus on Stage/Grade Features
focus_features = [
    col for col in comparison_df["feature"]
    if "Tumor Stage" in col or "Grade" in col
]

comparison_df[comparison_df["feature"].isin(focus_features)]

,feature,baseline_coef,baseline_hr,penalized_coef,penalized_hr
3,Neoplasm Histologic Grade_2.0,0.5260,1.6921,0.0257,1.0260
4,Neoplasm Histologic Grade_3.0,0.8279,2.2885,0.3256,1.3848
5,Tumor Stage_2.0,0.5127,1.6698,0.3295,1.3902
6,Tumor Stage_3.0,0.7509,2.1189,0.5323,1.7029
7,Tumor Stage_4.0,1.2290,3.4179,1.0042,2.7296


## Refinement Interpretation

This refinement step compares the baseline Cox PH model
with a penalized Cox PH model.

Key points:
- Train / validation split reduces overly optimistic evaluation.
- The penalized model is introduced to improve coefficient stability.
- Validation concordance is the main selection criterion.
- Overfitting gap helps assess generalization stability.

---

## 개선 단계 해석

이번 refinement 단계에서는 baseline Cox PH 모델과 penalized Cox PH 모델을 비교하였다.

핵심 의미:
- train / validation 분리를 통해 과도하게 낙관적인 평가를 줄였다.
- penalized 모델을 도입하여 계수 안정성을 높이고자 하였다.
- validation concordance를 주된 선택 기준으로 사용하였다.
- overfitting gap을 통해 일반화 안정성을 함께 점검하였다.

In [88]:
# 13. Save Results
model_performance.to_csv(
    "../data/processed/model_refinement_comparison.csv",
    index=False
)

comparison_df.to_csv(
    "../data/processed/model_refinement_coefficients.csv",
    index=False
)

print("Saved refinement outputs.")

Saved refinement outputs.


## Refinement Summary

This refinement step extends the baseline survival modeling workflow by:
- introducing train / validation evaluation
- comparing baseline and penalized Cox models
- checking coefficient stability for clinically relevant features
- selecting a more robust model based on validation performance

This helps move the project from a baseline analytical workflow
toward a more reliable and reproducible survival modeling pipeline.

---

## 개선 단계 요약

이번 refinement 단계에서는 다음을 추가하였다.
- train / validation 기반 평가
- baseline Cox와 penalized Cox 비교
- 임상적으로 중요한 변수의 계수 안정성 점검
- validation 성능을 기반으로 한 더 robust한 모델 선택

이를 통해 프로젝트는 단순 baseline 분석을 넘어
보다 신뢰 가능하고 재현 가능한 survival modeling pipeline으로 확장된다.